# Week 2: Improved Bi-LSTM NER Model

**LexiScan-Auto Named Entity Recognition (NER) System**

This notebook implements an improved Bi-LSTM model for extracting **DATE** and **MONEY** entities from legal documents using:
- **BIO Tagging**: 5 classes (O, B-DATE, I-DATE, B-MONEY, I-MONEY) for boundary-aware detection
- **GloVe Embeddings**: 100D pretrained vectors from Kaggle
- **Class Weighting**: Per-token balancing (O=0.3, B-*=3.0, I-*=2.5)
- **Two-Phase Training**: Freeze GloVe weights during warmup, then fine-tune
- **Threshold Tuning**: Optimal classification threshold for improved F1 score

**Environment**: TensorFlow 2.19.0, Python 3.12.12, Windows

In [ ]:
# CELL 1: LIBRARIES & IMPORTS
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Embedding, Dropout, Input, TimeDistributed
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import Callback, ReduceLROnPlateau
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_curve
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
import os
import re
import random

# Try to import seqeval for proper NER evaluation
try:
    from seqeval.metrics import classification_report, f1_score as seqeval_f1
    SEQEVAL_AVAILABLE = True
except ImportError:
    SEQEVAL_AVAILABLE = False

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")
if SEQEVAL_AVAILABLE:
    print("✅ seqeval available for proper NER metrics")
print("✅ Cell 1 complete: Libraries imported")

TensorFlow version: 2.19.0
NumPy version: 2.0.2
GPU Available: False
✅ Cell 1 complete: Libraries imported


In [ ]:
# CELL 2: GLOVE EMBEDDINGS VIA KAGGLEHUB
import kagglehub

print("Downloading GloVe embeddings from Kaggle...")
glove_path = kagglehub.dataset_download("danielwillgeorge/glove6b100dtxt")
glove_file = os.path.join(glove_path, "glove.6B.100d.txt")

def load_glove_embeddings(glove_file, embedding_dim=100):
    embeddings = {}
    with open(glove_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]
            vector = np.array(parts[1:], dtype=np.float32)
            if len(vector) == embedding_dim:
                embeddings[word] = vector
    return embeddings

glove_embeddings = load_glove_embeddings(glove_file)
EMBEDDING_DIM = 100

print(f"✅ Loaded {len(glove_embeddings):,} GloVe vectors (100D)")
print(f"   File: {glove_file}")
print("✅ Cell 2 complete: GloVe embeddings ready")

Using Colab cache for faster access to the 'glove6b100dtxt' dataset.
✅ Loaded 400,000 GloVe vectors (100D)
   File: /kaggle/input/glove6b100dtxt/glove.6B.100d.txt
✅ Cell 2 complete: GloVe embeddings ready


In [ ]:
# CELL 3: BIO TAGGING + EXPANDED TRAINING DATA (50+ SAMPLES)
BIO_TAGS = ['O', 'B-DATE', 'I-DATE', 'B-MONEY', 'I-MONEY']
tag2idx = {tag: idx for idx, tag in enumerate(BIO_TAGS)}
idx2tag = {idx: tag for tag, idx in tag2idx.items()}

print("BIO tag mapping:")
for tag, idx in tag2idx.items():
    print(f"  {tag:10} → {idx}")

# EXPANDED TRAINING DATA: 50+ diverse legal document samples + NEGATIVE EXAMPLES
sample_data = [
    # POSITIVE EXAMPLES (DATE/MONEY)
    (['The', 'meeting', 'is', 'on', 'January', '15', '2024'], ['O', 'O', 'O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Price', 'is', '$', '50'], ['O', 'O', 'B-MONEY', 'I-MONEY']),
    (['Due', 'date', 'is', 'March', '1'], ['O', 'O', 'O', 'B-DATE', 'I-DATE']),
    (['Cost', '$', '100', 'on', 'Feb', '28', '2024'], ['O', 'B-MONEY', 'I-MONEY', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Deadline', 'April', '10'], ['O', 'B-DATE', 'I-DATE']),
    (['The', 'contract', 'value', 'is', '$', '5000'], ['O', 'O', 'O', 'O', 'B-MONEY', 'I-MONEY']),
    (['Submit', 'by', 'December', '31', '2024'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Payment', 'of', '$', '1500', 'due', 'June', '1'], ['O', 'O', 'B-MONEY', 'I-MONEY', 'O', 'B-DATE', 'I-DATE']),
    (['Invoice', 'dated', 'March', '15', '2023'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Budget', 'approved', '$', '250', 'on', 'July', '4'], ['O', 'O', 'B-MONEY', 'I-MONEY', 'O', 'B-DATE', 'I-DATE']),
    (['Agreement', 'effective', 'September', '1', '2023'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Payment', 'amount', '$', '7500', 'USD'], ['O', 'O', 'B-MONEY', 'I-MONEY', 'I-MONEY']),
    (['Contract', 'expires', 'on', 'October', '31', '2024'], ['O', 'O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Total', 'cost', '$', '15000'], ['O', 'O', 'B-MONEY', 'I-MONEY']),
    (['Meeting', 'scheduled', 'for', 'November', '5'], ['O', 'O', 'O', 'B-DATE', 'I-DATE']),
    (['Invoice', 'amount', '$', '3200', 'due', 'immediately'], ['O', 'O', 'B-MONEY', 'I-MONEY', 'O', 'O']),
    (['The', 'project', 'starts', 'on', 'January', '10', '2025'], ['O', 'O', 'O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Fee', 'of', '$', '500', 'per', 'month'], ['O', 'O', 'B-MONEY', 'I-MONEY', 'O', 'O']),
    (['Warranty', 'valid', 'until', 'December', '15', '2023'], ['O', 'O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Deposit', '$', '1000', 'required'], ['O', 'B-MONEY', 'I-MONEY', 'O']),
    (['Effective', 'date', 'is', 'February', '1'], ['O', 'O', 'O', 'B-DATE', 'I-DATE']),
    (['Budget', 'allocation', '$', '50000', 'annually'], ['O', 'O', 'B-MONEY', 'I-MONEY', 'O']),
    (['Deadline', 'is', 'May', '30', '2024'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Amount', 'paid', '$', '2500', 'on', 'date'], ['O', 'O', 'B-MONEY', 'I-MONEY', 'O', 'O']),
    (['Event', 'on', 'August', '20'], ['O', 'O', 'B-DATE', 'I-DATE']),
    (['Price', 'tag', '$', '899', 'per', 'unit'], ['O', 'O', 'B-MONEY', 'I-MONEY', 'O', 'O']),
    (['Anniversary', 'date', 'June', '15', '2024'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Contract', 'value', '$', '25000', 'total'], ['O', 'O', 'B-MONEY', 'I-MONEY', 'O']),
    (['Action', 'date', 'March', '10', '2024'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Premium', '$', '450', 'monthly'], ['O', 'B-MONEY', 'I-MONEY', 'O']),
    (['Conference', 'on', 'July', '21', '2023'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Investment', '$', '100000', 'required'], ['O', 'B-MONEY', 'I-MONEY', 'O']),
    (['Launch', 'date', 'April', '1'], ['O', 'O', 'B-DATE', 'I-DATE']),
    (['Cost', 'estimate', '$', '12000'], ['O', 'O', 'B-MONEY', 'I-MONEY']),
    (['Opening', 'on', 'September', '15', '2024'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Salary', '$', '80000', 'annually'], ['O', 'B-MONEY', 'I-MONEY', 'O']),
    (['Check', 'in', 'date', 'November', '1'], ['O', 'O', 'O', 'B-DATE', 'I-DATE']),
    (['Rent', '$', '1500', 'monthly'], ['O', 'B-MONEY', 'I-MONEY', 'O']),
    (['Review', 'scheduled', 'January', '25', '2024'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Fine', '$', '200', 'payment'], ['O', 'B-MONEY', 'I-MONEY', 'O']),
    (['Closing', 'date', 'May', '15', '2024'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Bonus', '$', '5000', 'awarded'], ['O', 'B-MONEY', 'I-MONEY', 'O']),
    (['Termination', 'date', 'December', '31', '2023'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Settlement', '$', '75000', 'total'], ['O', 'B-MONEY', 'I-MONEY', 'O']),
    (['Renewal', 'on', 'August', '10'], ['O', 'O', 'B-DATE', 'I-DATE']),
    (['Invoice', '$', '4500', 'payable'], ['O', 'B-MONEY', 'I-MONEY', 'O']),
    (['Publication', 'date', 'June', '30', '2024'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    (['Rebate', '$', '300', 'available'], ['O', 'B-MONEY', 'I-MONEY', 'O']),
    (['Trial', 'period', 'starts', 'February', '14'], ['O', 'O', 'O', 'B-DATE', 'I-DATE']),
    (['Commission', '$', '10000', 'earned'], ['O', 'B-MONEY', 'I-MONEY', 'O']),
    # NEGATIVE EXAMPLES (no DATE/MONEY)
    (['The', 'document', 'is', 'signed'], ['O', 'O', 'O', 'O']),
    (['Please', 'review', 'the', 'agreement'], ['O', 'O', 'O', 'O']),
    (['We', 'confirm', 'the', 'terms'], ['O', 'O', 'O', 'O']),
    (['This', 'is', 'a', 'contract'], ['O', 'O', 'O', 'O']),
    (['All', 'parties', 'agree'], ['O', 'O', 'O']),
    (['The', 'agreement', 'is', 'final'], ['O', 'O', 'O', 'O']),
    (['We', 'need', 'your', 'approval'], ['O', 'O', 'O', 'O']),
    (['Please', 'sign', 'here'], ['O', 'O', 'O']),
    (['The', 'terms', 'apply'], ['O', 'O', 'O']),
    (['This', 'is', 'binding'], ['O', 'O', 'O']),
]

print(f"\n✅ Training samples: {len(sample_data)}")
print(f"   Example: {' '.join(sample_data[0][0])}")
print(f"   Tags:    {' '.join(sample_data[0][1])}")
print(f"   Coverage: {len(sample_data)} sentences with diverse entities")
print("✅ Cell 3 complete: Expanded BIO data ready")

BIO tag mapping:
  O          → 0
  B-DATE     → 1
  I-DATE     → 2
  B-MONEY    → 3
  I-MONEY    → 4

✅ Training samples: 60
   Example: The meeting is on January 15 2024
   Tags:    O O O O B-DATE I-DATE I-DATE
   Coverage: 60 sentences with diverse entities
✅ Cell 3 complete: Expanded BIO data ready


In [ ]:
# CELL 4: VOCABULARY (<PAD>=0, <UNK>=1, NO COLLISION)
all_tokens = set()
for sent_tokens, _ in sample_data:
    all_tokens.update(sent_tokens)

token2idx = {'<PAD>': 0, '<UNK>': 1}
for idx, token in enumerate(sorted(all_tokens), start=2):
    token2idx[token] = idx

idx2token = {idx: token for token, idx in token2idx.items()}
VOCAB_SIZE = len(token2idx)

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"  <PAD> index: {token2idx['<PAD>']} (zero vector, masked)")
print(f"  <UNK> index: {token2idx['<UNK>']} (mean GloVe vector)")
print(f"  Known tokens: indices 2–{VOCAB_SIZE-1}")
print(f"  Token sample: {list(sorted(all_tokens))[:5]}")
print("✅ Cell 4 complete: Vocabulary built")

Vocabulary size: 157
  <PAD> index: 0 (zero vector, masked)
  <UNK> index: 1 (mean GloVe vector)
  Known tokens: indices 2–156
  Token sample: ['$', '1', '10', '100', '1000']
✅ Cell 4 complete: Vocabulary built


In [ ]:
# CELL 5: TEXT PREPROCESSING + SEQUENCE PREPARATION + EMBEDDING MATRIX
MAX_SEQ_LENGTH = 20

def normalize_text(text):
    """Normalize text: lowercase, remove extra spaces."""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize_text(text):
    """Simple whitespace tokenization."""
    return text.split()

def prepare_sequences(data, token2idx, tag2idx, max_length=20):
    X, y = [], []
    for tokens, tags in data:
        token_ids = [token2idx.get(token, token2idx['<UNK>']) for token in tokens]
        tag_ids = [tag2idx[tag] for tag in tags]
        X.append(token_ids)
        y.append(tag_ids)
    X_padded = pad_sequences(X, maxlen=max_length, padding='post', value=token2idx['<PAD>'])
    y_padded = pad_sequences(y, maxlen=max_length, padding='post', value=tag2idx['O'])
    return X_padded, y_padded

# Prepare all data, then split into train/val/test (70/15/15 split)
X_all, y_all = prepare_sequences(sample_data, token2idx, tag2idx, MAX_SEQ_LENGTH)

train_split = int(0.7 * len(X_all))
val_split = int(0.85 * len(X_all))

X_train, y_train = X_all[:train_split], y_all[:train_split]
X_val, y_val = X_all[train_split:val_split], y_all[train_split:val_split]
X_test, y_test = X_all[val_split:], y_all[val_split:]

print(f"Total samples: {len(X_all)}")
print(f"Training set:   {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set:       {X_test.shape}")

def build_embedding_matrix(token2idx, glove_embeddings, embedding_dim=100):
    vocab_size = len(token2idx)
    all_vectors = np.array(list(glove_embeddings.values()), dtype=np.float32)
    unk_vector = all_vectors.mean(axis=0).astype(np.float32)
    embedding_matrix = np.zeros((vocab_size, embedding_dim), dtype=np.float32)
    embedding_matrix[token2idx['<PAD>']] = np.zeros(embedding_dim, dtype=np.float32)
    embedding_matrix[token2idx['<UNK>']] = unk_vector
    found, not_found = 0, 0
    for token, idx in token2idx.items():
        if token in ('<PAD>', '<UNK>'):
            continue
        token_lower = token.lower()
        if token_lower in glove_embeddings:
            embedding_matrix[idx] = glove_embeddings[token_lower]
            found += 1
        else:
            embedding_matrix[idx] = unk_vector
            not_found += 1
    print(f"\nEmbedding matrix: {embedding_matrix.shape}")
    print(f"  GloVe hits: {found}")
    print(f"  Fallback to <UNK>: {not_found}")
    return embedding_matrix

embedding_matrix = build_embedding_matrix(token2idx, glove_embeddings)
print("✅ Cell 5 complete: Preprocessing & splits ready")

Total samples: 60
Training set:   (42, 20)
Validation set: (9, 20)
Test set:       (9, 20)

Embedding matrix: (157, 100)
  GloVe hits: 154
  Fallback to <UNK>: 1
✅ Cell 5 complete: Preprocessing & splits ready


In [ ]:
# CELL 6: CLASS WEIGHTS VIA SAMPLE_WEIGHT
CLASS_WEIGHTS = {
    tag2idx['O']:       0.3,
    tag2idx['B-DATE']:  3.0,
    tag2idx['I-DATE']:  2.5,
    tag2idx['B-MONEY']: 3.0,
    tag2idx['I-MONEY']: 2.5,
}

def build_sample_weights(y_padded, class_weights):
    weights = np.ones_like(y_padded, dtype=np.float32)
    for tag_idx, weight in class_weights.items():
        weights[y_padded == tag_idx] = weight
    return weights

sample_weights = build_sample_weights(y_train, CLASS_WEIGHTS)

print("Class weight distribution:")
for tag, idx in sorted(tag2idx.items(), key=lambda x: x[1]):
    count = np.sum(y_train == idx)
    weight = CLASS_WEIGHTS.get(idx, 1.0)
    print(f"  {tag:10} (idx={idx}): count={count:3d}, weight={weight}")
print(f"\nSample weight matrix shape: {sample_weights.shape}")
print("✅ Cell 6 complete: Class weights built")

Class weight distribution:
  O          (idx=0): count=734, weight=0.3
  B-DATE     (idx=1): count= 24, weight=3.0
  I-DATE     (idx=2): count= 39, weight=2.5
  B-MONEY    (idx=3): count= 21, weight=3.0
  I-MONEY    (idx=4): count= 22, weight=2.5

Sample weight matrix shape: (42, 20)
✅ Cell 6 complete: Class weights built


In [ ]:
# CELL 7: CUSTOM F1EARLYSTOPPING CALLBACK
class F1EarlyStopping(Callback):
    def __init__(self, validation_data, patience=10):
        super().__init__()
        self.validation_data = validation_data
        self.patience = patience
        self.best_f1 = 0.0
        self.wait = 0
        self.best_weights = None
        self.history_f1 = []

    def on_epoch_end(self, epoch, logs=None):
        X_val, y_val = self.validation_data
        y_pred_probs = self.model.predict(X_val, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=-1)
        y_true_flat, y_pred_flat = [], []
        for i in range(len(y_val)):
            for j in range(len(y_val[i])):
                true_tag = y_val[i][j]
                pred_tag = y_pred[i][j]
                if true_tag != tag2idx['O'] or pred_tag != tag2idx['O']:
                    y_true_flat.append(1 if true_tag != tag2idx['O'] else 0)
                    y_pred_flat.append(1 if pred_tag != tag2idx['O'] else 0)
        current_f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0) if y_true_flat else 0.0
        self.history_f1.append(current_f1)
        if (epoch + 1) % 5 == 0:
            print(f"    Epoch {epoch+1:2d}: F1={current_f1:.4f} (best={self.best_f1:.4f})")
        if current_f1 > self.best_f1:
            self.best_f1 = current_f1
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n    Early stop at epoch {epoch+1}. Best F1: {self.best_f1:.4f}")
                self.model.stop_training = True
                if self.best_weights:
                    self.model.set_weights(self.best_weights)

print("✅ F1EarlyStopping callback defined")
print("✅ Cell 7 complete: Callback ready")

✅ F1EarlyStopping callback defined
✅ Cell 7 complete: Callback ready


In [ ]:
# CELL 8: IMPROVED BILSTM MODEL (Functional API)
def build_improved_bilstm(vocab_size, embedding_dim, embedding_matrix, lstm_units, num_tags, max_length):
    inputs = Input(shape=(max_length,), name='token_input')
    x = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        embeddings_initializer=keras.initializers.Constant(embedding_matrix),
        trainable=False,
        mask_zero=True,
        name='glove_embedding'
    )(inputs)
    x = Bidirectional(
        LSTM(lstm_units, return_sequences=True, dropout=0.3, recurrent_dropout=0.2),
        name='bilstm_1'
    )(x)
    x = Dropout(0.3, name='dropout_1')(x)
    x = Bidirectional(
        LSTM(lstm_units // 2, return_sequences=True, dropout=0.3, recurrent_dropout=0.2),
        name='bilstm_2'
    )(x)
    x = Dropout(0.3, name='dropout_2')(x)
    x = TimeDistributed(Dense(64, activation='relu'), name='dense_hidden')(x)
    x = Dropout(0.2, name='dropout_3')(x)
    outputs = TimeDistributed(Dense(num_tags, activation='softmax'), name='ner_output')(x)
    model = Model(inputs=inputs, outputs=outputs, name='BiLSTM_NER')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

NUM_TAGS = len(BIO_TAGS)
LSTM_UNITS = 64
model = build_improved_bilstm(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    embedding_matrix=embedding_matrix,
    lstm_units=LSTM_UNITS,
    num_tags=NUM_TAGS,
    max_length=MAX_SEQ_LENGTH
)
print("Model architecture:")
model.summary()
print("✅ Cell 8 complete: Model built")

Model architecture:


Model: "BiLSTM_NER"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ token_input         │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ glove_embedding     │ (None, 20, 100)   │     15,700 │ token_input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 20)        │          0 │ token_input[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 20, 128)   │     84,480 │ glove_embedding[… │
│ (Bidirectional)     │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 20, 128)   │          0 │ bilstm_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_2            │ (None, 20, 64)    │     41,216 │ dropout_1[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 20, 64)    │          0 │ bilstm_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_hidden        │ (None, 20, 64)    │      4,160 │ dropout_2[0][0],  │
│ (TimeDistributed)   │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 20, 64)    │          0 │ dense_hidden[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ner_output          │ (None, 20, 5)     │        325 │ dropout_3[0][0],  │
│ (TimeDistributed)   │                   │            │ not_equal_2[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 145,881 (569.85 KB)

 Trainable params: 130,181 (508.52 KB)

 Non-trainable params: 15,700 (61.33 KB)

✅ Cell 8 complete: Model built


In [ ]:
# CELL 9: TWO-PHASE TRAINING (Freeze GloVe → Unfreeze)
# Use proper 70/15/15 train/val/test splits from Cell 5
sw_train = build_sample_weights(y_train, CLASS_WEIGHTS)

print(f"Training set:   {X_train.shape} with weights {sw_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set:       {X_test.shape}")

f1_callback = F1EarlyStopping(validation_data=(X_val, y_val), patience=10)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=0)

print("\n" + "="*60)
print("PHASE 1: Warmup — GloVe frozen")
print("="*60)

batch_size = max(2, len(X_train) // 8)
history_phase1 = model.fit(
    X_train, y_train,
    sample_weight=sw_train,
    epochs=15,
    batch_size=batch_size,
    validation_data=(X_val, y_val),
    callbacks=[f1_callback, lr_scheduler],
    verbose=0
)
print(f"✅ Phase 1 complete. Best F1: {f1_callback.best_f1:.4f}")

print("\n" + "="*60)
print("PHASE 2: Fine-tuning — GloVe unfrozen")
print("="*60)

model.get_layer('glove_embedding').trainable = True
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
f1_callback.wait = 0

history_phase2 = model.fit(
    X_train, y_train,
    sample_weight=sw_train,
    epochs=40,
    batch_size=batch_size,
    validation_data=(X_val, y_val),
    callbacks=[f1_callback, lr_scheduler],
    verbose=0
)
print(f"✅ Phase 2 complete. Best F1: {f1_callback.best_f1:.4f}")
print("✅ Cell 9 complete: Training done")

Training set:   (42, 20) with weights (42, 20)
Validation set: (9, 20)
Test set:       (9, 20)

PHASE 1: Warmup — GloVe frozen
    Epoch  5: F1=0.1818 (best=0.1818)
    Epoch 10: F1=0.1827 (best=0.1827)
    Epoch 15: F1=0.1885 (best=0.1885)
✅ Phase 1 complete. Best F1: 0.1885

PHASE 2: Fine-tuning — GloVe unfrozen
    Epoch  5: F1=0.8000 (best=0.7826)
    Epoch 10: F1=0.8780 (best=0.8780)
    Epoch 15: F1=0.8780 (best=0.8780)

    Early stop at epoch 17. Best F1: 0.8780
✅ Phase 2 complete. Best F1: 0.8780
✅ Cell 9 complete: Training done


In [ ]:
# CELL 10: COMPREHENSIVE EVALUATION + THRESHOLD TUNING + SEQEVAL
def evaluate_bio_model(y_true, y_pred, tag2idx, idx2tag, dataset_name="Dataset"):
    print(f"\n{'='*65}")
    print(f"EVALUATION — {dataset_name}")
    print(f"{'='*65}")
    y_true_flat, y_pred_flat = [], []
    for i in range(len(y_true)):
        for j in range(len(y_true[i])):
            y_true_flat.append(y_true[i][j])
            y_pred_flat.append(y_pred[i][j])
    y_true_flat = np.array(y_true_flat)
    y_pred_flat = np.array(y_pred_flat)
    y_true_bin = (y_true_flat != tag2idx['O']).astype(int)
    y_pred_bin = (y_pred_flat != tag2idx['O']).astype(int)
    f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)
    prec = precision_score(y_true_bin, y_pred_bin, zero_division=0)
    rec = recall_score(y_true_bin, y_pred_bin, zero_division=0)
    print(f"  Overall Entity F1:        {f1:.4f}")
    print(f"  Overall Entity Precision: {prec:.4f}")
    print(f"  Overall Entity Recall:    {rec:.4f}")
    print(f"\n  Per-tag breakdown:")
    for tag, idx in tag2idx.items():
        if tag == 'O':
            continue
        y_t = (y_true_flat == idx).astype(int)
        y_p = (y_pred_flat == idx).astype(int)
        tag_f1 = f1_score(y_t, y_p, zero_division=0)
        tag_prec = precision_score(y_t, y_p, zero_division=0)
        tag_rec = recall_score(y_t, y_p, zero_division=0)
        print(f"    {tag:10} | F1: {tag_f1:.4f} | Prec: {tag_prec:.4f} | Rec: {tag_rec:.4f}")
    return f1, prec, rec

def tune_threshold(model, X_val, y_val, tag2idx):
    y_pred_probs = model.predict(X_val, verbose=0)
    y_true_bin, y_score = [], []
    for i in range(len(y_val)):
        for j in range(len(y_val[i])):
            true_tag = y_val[i][j]
            # Use OVERALL max probability, not just entity probability
            max_prob = np.max(y_pred_probs[i][j])
            y_true_bin.append(1 if true_tag != tag2idx['O'] else 0)
            y_score.append(max_prob)
    y_true_bin = np.array(y_true_bin)
    y_score = np.array(y_score)
    # Broader threshold search for better precision
    thresholds = np.arange(0.2, 0.99, 0.03)
    f1_scores = []
    for thresh in thresholds:
        y_pred_bin = (y_score >= thresh).astype(int)
        f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)
        f1_scores.append(f1)
    best_idx = np.argmax(f1_scores)
    best_thresh = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    precision_curve, recall_curve, _ = precision_recall_curve(y_true_bin, y_score)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(thresholds, f1_scores, 'b-o', markersize=6, linewidth=2)
    plt.axvline(x=best_thresh, color='r', linestyle='--', linewidth=2, label=f'Optimal: {best_thresh:.2f}')
    plt.xlabel('Threshold')
    plt.ylabel('F1 Score')
    plt.title('F1 vs Classification Threshold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.subplot(1, 2, 2)
    plt.plot(recall_curve, precision_curve, 'g-', linewidth=2)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plot_path = os.path.join(os.getcwd(), 'threshold_tuning.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\n✅ Plot saved: {plot_path}")
    return best_thresh, best_f1

# Predict on all three sets
y_pred_probs_train = model.predict(X_train, verbose=0)
y_pred_train = np.argmax(y_pred_probs_train, axis=-1)
y_pred_probs_val = model.predict(X_val, verbose=0)
y_pred_val = np.argmax(y_pred_probs_val, axis=-1)
y_pred_probs_test = model.predict(X_test, verbose=0)
y_pred_test = np.argmax(y_pred_probs_test, axis=-1)

# Standard token-level evaluation
f1_tr, prec_tr, rec_tr = evaluate_bio_model(y_train, y_pred_train, tag2idx, idx2tag, "Training Set")
f1_val, prec_val, rec_val = evaluate_bio_model(y_val, y_pred_val, tag2idx, idx2tag, "Validation Set")
f1_test, prec_test, rec_test = evaluate_bio_model(y_test, y_pred_test, tag2idx, idx2tag, "Test Set")

# Threshold tuning
best_threshold, best_val_f1 = tune_threshold(model, X_val, y_val, tag2idx)

# Seqeval evaluation (proper NER metric)
if SEQEVAL_AVAILABLE:
    print("\n" + "="*65)
    print("SEQEVAL ENTITY-LEVEL METRICS (Proper NER Evaluation)")
    print("="*65)
    
    def get_seqeval_format(y_true, y_pred, idx2tag, tag2idx):
        "Convert sequences to seqeval format (list of lists of tags)"
        y_true_tags, y_pred_tags = [], []
        for i in range(len(y_true)):
            true_seq, pred_seq = [], []
            for j in range(len(y_true[i])):
                if y_true[i][j] != tag2idx['O'] or y_pred[i][j] != tag2idx['O']:
                    true_seq.append(idx2tag[y_true[i][j]])
                    pred_seq.append(idx2tag[y_pred[i][j]])
            if true_seq:
                y_true_tags.append(true_seq)
                y_pred_tags.append(pred_seq)
        return y_true_tags, y_pred_tags
    
    y_true_seq_test, y_pred_seq_test = get_seqeval_format(y_test, y_pred_test, idx2tag, tag2idx)
    if y_true_seq_test:
        print("\nTest Set Classification Report:")
        print(classification_report(y_true_seq_test, y_pred_seq_test))

print("✅ Cell 10 complete: Comprehensive evaluation done")


EVALUATION — Training Set
  Overall Entity F1:        0.8870
  Overall Entity Precision: 0.7970
  Overall Entity Recall:    1.0000

  Per-tag breakdown:
    B-DATE     | F1: 0.9231 | Prec: 0.8571 | Rec: 1.0000
    I-DATE     | F1: 1.0000 | Prec: 1.0000 | Rec: 1.0000
    B-MONEY    | F1: 0.8235 | Prec: 0.7000 | Rec: 1.0000
    I-MONEY    | F1: 0.7586 | Prec: 0.6111 | Rec: 1.0000

EVALUATION — Validation Set
  Overall Entity F1:        0.8780
  Overall Entity Precision: 0.7826
  Overall Entity Recall:    1.0000

  Per-tag breakdown:
    B-DATE     | F1: 0.8889 | Prec: 0.8000 | Rec: 1.0000
    I-DATE     | F1: 1.0000 | Prec: 1.0000 | Rec: 1.0000
    B-MONEY    | F1: 1.0000 | Prec: 1.0000 | Rec: 1.0000
    I-MONEY    | F1: 0.6667 | Prec: 0.5000 | Rec: 1.0000

EVALUATION — Test Set
  Overall Entity F1:        0.0000
  Overall Entity Precision: 0.0000
  Overall Entity Recall:    0.0000

  Per-tag breakdown:
    B-DATE     | F1: 0.0000 | Prec: 0.0000 | Rec: 0.0000
    I-DATE     | F1: 0.0000

In [ ]:
# CELL 11: INFERENCE + TEST CASES + FINAL SUMMARY
def predict_ner(text, model, token2idx, idx2tag, tag2idx, max_length=20, threshold=None):
    """
    Predict NER tags for text with sliding window for long sentences.
    
    FIXES FOR LONG SENTENCES:
    - Sentences ≤ max_length: single pass prediction
    - Sentences > max_length: overlapping chunks (chunk_size=max_length, overlap=5)
    - For overlapping tokens: keep prediction with higher confidence score
    
    Args:
        text: Input text string
        model: Trained NER model
        token2idx: Token to index dictionary
        idx2tag: Index to tag dictionary
        tag2idx: Tag to index dictionary
        max_length: Maximum sequence length (default 20)
        threshold: Optional confidence threshold for entity detection
    
    Returns:
        List of (token, tag) tuples for all input tokens
    """
    tokens = text.split()
    
    # ─── CASE 1: SHORT SENTENCE (≤ max_length) ─────────────────────────
    if len(tokens) <= max_length:
        token_ids = [token2idx.get(token, token2idx['<UNK>']) for token in tokens]
        X = pad_sequences([token_ids], maxlen=max_length, padding='post', value=token2idx['<PAD>'])
        probs = model.predict(X, verbose=0)[0]
        
        result = []
        for i, token in enumerate(tokens):
            if threshold is not None:
                max_prob = np.max(probs[i])
                if max_prob >= threshold:
                    tag_idx = np.argmax(probs[i])
                else:
                    tag_idx = tag2idx['O']
            else:
                tag_idx = np.argmax(probs[i])
            result.append((token, idx2tag[tag_idx]))
        return result
    
    # ─── CASE 2: LONG SENTENCE (> max_length) - SLIDING WINDOW ─────────
    overlap = 5  # Overlap between chunks
    stride = max_length - overlap  # Step size for next chunk
    
    # Store best prediction for each token (to handle overlaps)
    token_predictions = {}  # {token_idx: (tag_idx, confidence_score)}
    
    # Generate chunk start positions
    chunk_starts = list(range(0, len(tokens), stride))
    # Ensure last chunk covers remaining tokens (don't leave gap at end)
    if chunk_starts[-1] + max_length < len(tokens):
        chunk_starts.append(len(tokens) - max_length)
    # Remove duplicates and sort
    chunk_starts = sorted(set(chunk_starts))
    
    # ─── Process each chunk ──────────────────────────────────────────────
    for chunk_idx, chunk_start in enumerate(chunk_starts):
        chunk_end = min(chunk_start + max_length, len(tokens))
        chunk_tokens = tokens[chunk_start:chunk_end]
        
        # Convert tokens to IDs and pad
        token_ids = [token2idx.get(token, token2idx['<UNK>']) for token in chunk_tokens]
        X = pad_sequences([token_ids], maxlen=max_length, padding='post', value=token2idx['<PAD>'])
        probs = model.predict(X, verbose=0)[0]
        
        # Extract predictions for this chunk
        for local_idx, token in enumerate(chunk_tokens):
            global_idx = chunk_start + local_idx
            max_prob = np.max(probs[local_idx])
            tag_idx = np.argmax(probs[local_idx])
            
            # Apply threshold if specified
            if threshold is not None and max_prob < threshold:
                tag_idx = tag2idx['O']
            
            # ─── OVERLAP RESOLUTION ──────────────────────────────────────
            # For tokens in overlapping regions, keep prediction 
            # with higher confidence (max softmax score)
            if global_idx in token_predictions:
                _, prev_confidence = token_predictions[global_idx]
                if max_prob > prev_confidence:
                    token_predictions[global_idx] = (tag_idx, max_prob)
            else:
                token_predictions[global_idx] = (tag_idx, max_prob)
    
    # ─── Build final result (maintain original token order) ──────────────
    result = []
    for i, token in enumerate(tokens):
        tag_idx, _ = token_predictions[i]
        result.append((token, idx2tag[tag_idx]))
    
    return result

# Test cases from dataset + new examples
test_cases = [
    "Payment due on January 30 2024",
    "Invoice for $ 1500 on March 15",
    "Deadline February 10 with budget $ 5000",
    "Meeting scheduled for April 1st costs $ 250",
    "Due on May 20 Budget $ 10000",
    "Contract signed December 15 2023 for $ 75000",
    "Annual fee of $ 1200 payable by June 30",
    "Settlement amount $ 45000 effective July 1 2024",
    "Warranty expires December 31 2023",
    "New pricing $ 999 per unit",
]

print("\n" + "="*70)
print(f"INFERENCE TEST CASES (threshold: {best_threshold:.2f})")
print("="*70)

for test_text in test_cases:
    predictions = predict_ner(test_text, model, token2idx, idx2tag, tag2idx, threshold=best_threshold)
    print(f"\n► {test_text}")
    entities = [(tok, tag) for tok, tag in predictions if tag != 'O']
    if entities:
        for tok, tag in entities:
            print(f"    {tok:20} → {tag}")
    else:
        print("    (no entities detected)")

print(f"""
╔══════════════════════════════════════════════════════════════════════════╗
║                    FINAL TRAINING SUMMARY                               ║
╚══════════════════════════════════════════════════════════════════════════╝

DATASET:
  • Total samples: {len(sample_data)}
  • Training: {len(X_train)} | Validation: {len(X_val)} | Test: {len(X_test)}
  • Split ratio: 70% / 15% / 15%

MODEL: Improved Bi-LSTM NER with GloVe + BIO Tagging

ARCHITECTURE:
  • GloVe 100D embeddings (frozen warmup, then fine-tuned)
  • Bidirectional LSTM Layer 1: {LSTM_UNITS} units
  • Bidirectional LSTM Layer 2: {LSTM_UNITS // 2} units
  • Per-token classification via TimeDistributed Dense

TRAINING:
  • Phase 1: Freeze GloVe, train LSTM (15 epochs)
  • Phase 2: Unfreeze GloVe, fine-tune (40 epochs)
  • Class weights: O={CLASS_WEIGHTS[tag2idx['O']]}, B-*={CLASS_WEIGHTS[tag2idx['B-DATE']]}, I-*={CLASS_WEIGHTS[tag2idx['I-DATE']]}
  • Dropout regularization: 0.2-0.3
  • Early stopping: F1-based (patience=10)

RESULTS:
  Training   → F1={f1_tr:.4f} | Precision={prec_tr:.4f} | Recall={rec_tr:.4f}
  Validation → F1={f1_val:.4f} | Precision={prec_val:.4f} | Recall={rec_val:.4f}
  Test       → F1={f1_test:.4f} | Precision={prec_test:.4f} | Recall={rec_test:.4f}
  
THRESHOLD TUNING:
  • Optimal threshold: {best_threshold:.2f}
  • Best validation F1: {best_val_f1:.4f}

STATUS: ✅ Model trained and ready for evaluation

""")

print("\n✅✅✅ COMPLETE NOTEBOOK EXECUTION FINISHED ✅✅✅")

print("✅ Cell 11 complete: Inference and summary done")


INFERENCE TEST CASES (threshold: 0.92)

► Payment due on January 30 2024
    January              → B-DATE
    30                   → I-DATE
    2024                 → I-DATE

► Invoice for $ 1500 on March 15
    $                    → B-MONEY
    March                → B-DATE
    15                   → I-DATE

► Deadline February 10 with budget $ 5000
    February             → B-DATE
    10                   → I-DATE
    with                 → I-DATE
    budget               → I-DATE
    $                    → I-DATE
    5000                 → I-DATE

► Meeting scheduled for April 1st costs $ 250
    April                → B-DATE
    1st                  → I-DATE
    costs                → I-DATE
    $                    → I-DATE
    250                  → I-DATE

► Due on May 20 Budget $ 10000
    20                   → I-DATE
    Budget               → I-DATE
    $                    → I-DATE
    10000                → I-DATE

► Contract signed December 15 2023 for $ 75000
    Dec

In [ ]:
# Try to find the contracts folder
import glob

possible_paths = [
    r"C:\Users\wadje\Downloads\500_contracts_25_templates_Part2",
    r"C:\Users\wadje\Downloads\500_diverse_contracts",
    "C:\\Users\\wadje\\Downloads\\500_contracts_25_templates_Part2",
    "C:\\Users\\wadje\\Downloads\\500_diverse_contracts",
]

# Also try with forward slashes
import pathlib
for p in list(possible_paths):
    possible_paths.append(str(pathlib.Path(p)))

pdf_files = []
CONTRACTS_FOLDER_FOUND = ""

for path in possible_paths:
    try:
        if os.path.exists(path):
            pdf_files = sorted([f for f in os.listdir(path) if f.endswith('.pdf')])
            CONTRACTS_FOLDER_FOUND = path
            print(f"✅ Found folder: {path}")
            print(f"   Contains {len(pdf_files)} PDF files")
            break
    except Exception as e:
        pass

if not CONTRACTS_FOLDER_FOUND:
    print(f"⚠️  Could not find any contract folders")
    print(f"    Creating synthetic dataset instead...")
    # Create synthetic data so tests can continue
    pdf_training_data = [
        (['Payment', 'of', '$', '5000', 'due', 'January', '15'], ['O', 'O', 'B-MONEY', 'I-MONEY', 'O', 'B-DATE', 'I-DATE']),
        (['Invoice', 'amount', 'is', '$', '1500', 'on', 'March', '1'], ['O', 'O', 'O', 'B-MONEY', 'I-MONEY', 'O', 'B-DATE', 'I-DATE']),
        (['The', 'contract', 'value', 'is', '$', '25000'], ['O', 'O', 'O', 'O', 'B-MONEY', 'I-MONEY']),
        (['Deadline', 'is', 'December', '31', '2024'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
        (['Budget', 'approved', 'for', '$', '10000', 'effective', 'February', '1'], ['O', 'O', 'O', 'B-MONEY', 'I-MONEY', 'O', 'B-DATE', 'I-DATE']),
    ]
    print(f"   Created {len(pdf_training_data)} synthetic samples")
    money_sentences = [x for x in pdf_training_data if any('MONEY' in t for t in x[1])]
    date_sentences = [x for x in pdf_training_data if any('DATE' in t for t in x[1])]
    negative_sentences = []
else:
    CONTRACTS_FOLDER = CONTRACTS_FOLDER_FOUND
    print(f"Found {len(pdf_files)} PDF files")
    print("Extracting sentences...")

    # ── Extract sentences from all PDFs ─────────────────────────
    money_sentences    = []
    date_sentences     = []
    negative_sentences = []

    seen_money    = []
    seen_dates    = []
    seen_negatives = []

    for pdf_file in pdf_files:
        filepath = os.path.join(CONTRACTS_FOLDER, pdf_file)
        try:
            with pdfplumber.open(filepath) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()
                    if not text:
                        continue

                    for line in text.split('\n'):
                        line = line.strip()

                        # Skip too short or too long
                        if len(line) < 15:
                            continue
                        tokens = smart_tokenize(line)
                        if len(tokens) < 4 or len(tokens) > 30:
                            continue

                        has_money = bool(money_pattern.search(line))
                        has_date  = bool(date_pattern.search(line))

                        # ── MONEY sentences (limit 300) ───────────
                        if has_money and not has_date and len(money_sentences) < 300:
                            if not is_duplicate(line, seen_money):
                                tags = auto_bio_tag(tokens)
                                if 'B-MONEY' in tags:
                                    money_sentences.append((tokens, tags))
                                    seen_money.append(line)

                        # ── DATE sentences (limit 300) ────────────
                        elif has_date and not has_money and len(date_sentences) < 300:
                            if not is_duplicate(line, seen_dates):
                                tags = auto_bio_tag(tokens)
                                if 'B-DATE' in tags:
                                    date_sentences.append((tokens, tags))
                                    seen_dates.append(line)

                        # ── NEGATIVE sentences (limit 150) ────────
                        elif not has_money and not has_date:
                            word_count = len(tokens)
                            if 5 <= word_count <= 25 and len(negative_sentences) < 150:
                                if not is_duplicate(line, seen_negatives):
                                    tags = ['O'] * len(tokens)
                                    negative_sentences.append((tokens, tags))
                                    seen_negatives.append(line)

        except Exception as e:
            continue

⚠️  Could not find any contract folders
    Creating synthetic dataset instead...
   Created 5 synthetic samples


In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# CHANGE 1: SMART TOKENIZATION FUNCTION - Enhanced for dates & money
# ───────────────────────────────────────────────────────────────────────────
def smart_tokenize(text):
    """Intelligent tokenization for contract documents with special money/date handling"""
    # Remove commas inside numbers: $4,433,490 → $4433490
    text = re.sub(r'(\d),(\d)', r'\1\2', text)
    
    # Separate $ from numbers: $5000 → $ 5000
    text = re.sub(r'\$(\d)', r'$ \1', text)
    
    # Split dates by various formats
    # Slash dates: 04/09/2027 → 04 09 2027
    text = re.sub(r'(\d{1,2})[\/](\d{1,2})[\/](\d{2,4})', r'\1 \2 \3', text)
    
    # ISO dates: 2026-03-04 → 2026 03 04
    text = re.sub(r'(\d{4})-(\d{2})-(\d{2})', r'\1 \2 \3', text)
    
    # Separate trailing punctuation from words
    text = re.sub(r"([a-zA-Z0-9])([',\.\)\"])", r'\1 \2', text)
    
    # Split and filter out standalone punctuation
    tokens = text.split()
    return [t for t in tokens if t not in [',', '.', ')', '"', "'"]]

# Test smart_tokenize
test_cases = [
    "$4,433,490 due on 04/09/2027",
    "February 11, 2026",
    "effective as of 2026-03-04"
]
print("Testing smart_tokenize...")
for test in test_cases:
    result = smart_tokenize(test)
    print(f"  Input:  {test}")
    print(f"  Output: {result}")
print()



Testing smart_tokenize...
  Input:  $4,433,490 due on 04/09/2027
  Output: ['$', '4433490', 'due', 'on', '04', '09', '2027']
  Input:  February 11, 2026
  Output: ['February', '11', '2026']
  Input:  effective as of 2026-03-04
  Output: ['effective', 'as', 'of', '2026', '03', '04']



In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 13: PDF DATA EXTRACTION - Extract DATE samples from 500 legal contracts
# ═════════════════════════════════════════════════════════════════════════════

print("\n" + "="*90)
print("📄 CELL 13: PDF DATA EXTRACTION FROM 500 CONTRACT DOCUMENTS")
print("="*90)

# Check if pdfplumber is available, install if needed
try:
    import pdfplumber
    print("✅ pdfplumber available")
except ImportError:
    print("⏳ Installing pdfplumber...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pdfplumber", "-q"])
    import pdfplumber
    print("✅ pdfplumber installed")

# ─── STEP 1: DEFINE DATE PATTERNS ────────────────────────────────────────────
date_patterns = [
    (r'\d{1,2}[\/\-]\d{1,2}[\/\-]\d{2,4}', 'DD/MM/YYYY'),          # 29/11/2019
    (r'\d{4}-\d{2}-\d{2}', 'ISO_YYYY-MM-DD'),                        # 2023-03-24
    (r'(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4}', 'Month_DD_YYYY'),  # January 19, 2025
    (r'\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\w*\s+\d{4}', 'DD_Mon_YYYY'),  # 15 Feb 2025
]

print(f"\n📋 STEP 1: Date patterns loaded ({len(date_patterns)} formats)")
for pattern, name in date_patterns:
    print(f"   • {name:20s} → {pattern[:50]}...")

# ─── STEP 2: DEFINE BIO TAGGING FUNCTION ────────────────────────────────────
def smart_tokenize_with_bio(text, date_patterns):
    """
    Tokenize text and automatically assign BIO tags for detected dates.
    Returns: (token_list, tag_list)
    """
    # Tokenize by whitespace
    tokens = text.split()
    tags = ['O'] * len(tokens)  # Start with all O (non-entity)
    
    # Find all date matches in the original text
    date_matches = []
    for pattern, pattern_name in date_patterns:
        for match in re.finditer(pattern, text, re.IGNORECASE):
            date_matches.append({
                'text': match.group(),
                'start': match.start(),
                'end': match.end(),
                'pattern': pattern_name
            })
    
    # If no dates found, return as-is
    if not date_matches:
        return tokens, tags
    
    # Sort matches by position
    date_matches = sorted(date_matches, key=lambda x: x['start'])
    
    # Reconstruct token positions to map to matches
    char_pos = 0
    token_start_pos = []  # Store character position of each token start
    
    for token in tokens:
        # Find where this token starts in the text
        start_pos = text.find(token, char_pos)
        if start_pos != -1:
            token_start_pos.append((start_pos, start_pos + len(token)))
            char_pos = start_pos + len(token)
        else:
            # Fallback if token not found
            token_start_pos.append((char_pos, char_pos))
    
    # Tag tokens that overlap with date matches
    for match in date_matches:
        match_start = match['start']
        match_end = match['end']
        
        # Find tokens that overlap with this match
        is_first_token = True
        for idx, (tok_start, tok_end) in enumerate(token_start_pos):
            # Check if token overlaps with match
            if tok_start < match_end and tok_end > match_start:
                if is_first_token:
                    tags[idx] = 'B-DATE'  # Beginning of entity
                    is_first_token = False
                else:
                    tags[idx] = 'I-DATE'  # Inside entity
    
    return tokens, tags

# ─── STEP 3: DEFINE NEGATIVE SENTENCE EXTRACTOR ──────────────────────────────
def extract_negative_sentences(pdf_text, num_negative=50, max_attempts=1000):
    """Extract sentences with no dates or money mentions."""
    sentences = re.split(r'[.!?\n]+', pdf_text)
    negative_sents = []
    money_pattern = r'\$|\busd\b|\bmoney\b|\bpayment\b|\bprice\b|\bcost\b|\bfee\b'
    
    for sent in sentences:
        if len(negative_sents) >= num_negative:
            break
        
        sent = sent.strip()
        if not sent or len(sent) < 10:  # Skip very short sentences
            continue
        
        # Check if sentence contains any date or money
        has_date = any(re.search(pattern[0], sent, re.IGNORECASE) for pattern in date_patterns)
        has_money = bool(re.search(money_pattern, sent, re.IGNORECASE))
        
        if not has_date and not has_money:
            negative_sents.append(sent)
    
    return negative_sents[:num_negative]

# ─── STEP 4: LOAD AND PROCESS PDFs ───────────────────────────────────────────
print(f"\n📂 STEP 2: Scanning PDF folder...")

pdf_folder = r"C:\Users\wadje\Downloads\500_diverse_contracts"
import os

if not os.path.exists(pdf_folder):
    print(f"⚠️  WARNING: Folder not found at {pdf_folder}")
    print(f"   Creating empty pdf_training_data with 0 samples from PDFs")
    pdf_training_data = [
        # Add a few synthetic examples as fallback
        (['Signed', 'on', '29/11/2019', 'by', 'both', 'parties'], 
         ['O', 'O', 'B-DATE', 'O', 'O', 'O']),
        (['Payment', 'due', 'on', '2023-03-24'], 
         ['O', 'O', 'O', 'B-DATE']),
        (['Contract', 'dated', 'January', '19', ',', '2025'], 
         ['O', 'O', 'B-DATE', 'I-DATE', 'O', 'I-DATE']),
        (['Effective', 'date', ':', '15', 'Feb', '2025'], 
         ['O', 'O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
    ]
    pdf_count = 0
    extraction_stats = {fmt: 0 for _, fmt in date_patterns}
else:
    # Get list of PDF files
    pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith('.pdf')]
    pdf_count = len(pdf_files)
    print(f"✅ Found {pdf_count} PDF files")
    
    # ─── STEP 5: EXTRACT TEXT AND DATE SENTENCES ─────────────────────────────
    print(f"\n🔍 STEP 3: Extracting date sentences from PDFs...")
    
    all_extracted_sentences = []
    extraction_stats = {fmt: 0 for _, fmt in date_patterns}
    failed_pdfs = 0
    successful_pdfs = 0
    
    for i, pdf_file in enumerate(pdf_files[:500]):  # Limit to 500
        try:
            pdf_path = os.path.join(pdf_folder, pdf_file)
            with pdfplumber.open(pdf_path) as pdf:
                text = ""
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + " "
            
            # Extract sentences containing dates
            sentences = re.split(r'[.!?\n]+', text)
            for sent in sentences:
                sent = sent.strip()
                if not sent or len(sent) < 15:
                    continue
                
                # Check if sentence matches any date pattern
                for pattern, fmt in date_patterns:
                    if re.search(pattern, sent, re.IGNORECASE):
                        all_extracted_sentences.append(sent)
                        extraction_stats[fmt] += 1
                        break  # Count once per sentence
            
            successful_pdfs += 1
            if (i + 1) % 50 == 0:
                print(f"   Processed {i+1}/{len(pdf_files[:500])} PDFs...")
        
        except Exception as e:
            failed_pdfs += 1
            if failed_pdfs <= 5:  # Show first 5 errors only
                print(f"   ⚠️  Error reading {pdf_file}: {str(e)[:50]}")
    
    print(f"✅ Processed {successful_pdfs} PDFs successfully ({failed_pdfs} failed)")
    print(f"\n   Date patterns extracted:")
    total_extracted = 0
    for fmt, count in extraction_stats.items():
        print(f"      • {fmt:20s}: {count:4d} sentences")
        total_extracted += count
    print(f"      {'─'*30}")
    print(f"      • {'TOTAL':20s}: {total_extracted:4d} sentences")
    
    # ─── STEP 6: DEDUPLICATE & AUTO-TAG SENTENCES ────────────────────────────
    print(f"\n🏷️  STEP 4: Deduplicating and auto-tagging sentences...")
    
    # Remove duplicates (case-insensitive) while preserving order
    seen = set()
    unique_sentences = []
    for sent in all_extracted_sentences:
        sent_lower = sent.lower()
        if sent_lower not in seen:
            seen.add(sent_lower)
            unique_sentences.append(sent)
    
    print(f"   Raw sentences: {len(all_extracted_sentences)}")
    print(f"   After deduplication: {len(unique_sentences)}")
    
    # Filter by length and convert to BIO format
    pdf_training_data = []
    skipped_long = 0
    
    for sent in unique_sentences:
        tokens, tags = smart_tokenize_with_bio(sent, date_patterns)
        
        # Skip if too long
        if len(tokens) > MAX_LENGTH:
            skipped_long += 1
            continue
        
        pdf_training_data.append((tokens, tags))
        
        if len(pdf_training_data) >= 300:  # Limit to 300 samples
            break
    
    print(f"   Valid sentences (≤{MAX_LENGTH} tokens): {len(pdf_training_data)}")
    print(f"   Skipped (too long): {skipped_long}")
    
    # ─── STEP 7: ADD NEGATIVE SENTENCES ──────────────────────────────────────
    print(f"\n➖ STEP 5: Extracting negative sentences (no dates/money)...")
    
    all_pdf_text = ""
    for i, pdf_file in enumerate(pdf_files[:100]):  # Use more PDFs for negatives
        try:
            pdf_path = os.path.join(pdf_folder, pdf_file)
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        all_pdf_text += page_text + " "
            if (i + 1) % 25 == 0:
                print(f"   Processed {i+1}/100 PDFs for negative extraction...")
        except:
            pass
    
    negative_sents = extract_negative_sentences(all_pdf_text, num_negative=50)
    print(f"   Extracted {len(negative_sents)} negative sentences")
    
    # Add negative sentences to training data
    for neg_sent in negative_sents:
        tokens = neg_sent.split()
        if 5 <= len(tokens) <= MAX_LENGTH:  # Reasonable length
            tags = ['O'] * len(tokens)  # All non-entity
            pdf_training_data.append((tokens, tags))
    
    print(f"   Added {len(negative_sents)} negative samples")

# ─── STEP 6: SUMMARY ─────────────────────────────────────────────────────────
print(f"\n{'='*90}")
print(f"📊 EXTRACTION SUMMARY")
print(f"{'='*90}\n")

print(f"PDFs processed: {pdf_count}")
print(f"PDF training data samples extracted: {len(pdf_training_data)}")
print(f"\nSample of extracted data:")

# Show a few examples
for idx, (tokens, tags) in enumerate(pdf_training_data[:3]):
    print(f"\n   Example {idx+1}:")
    print(f"   Tokens: {tokens}")
    print(f"   Tags:   {tags}")
    
    # Count entity types
    entity_types = set([tag.split('-')[1] for tag in tags if tag != 'O'])
    if entity_types:
        print(f"   Entities: {', '.join(sorted(entity_types))}")

# Vocabulary expansion
pdf_vocab = set()
for tokens, _ in pdf_training_data:
    pdf_vocab.update(tokens)

print(f"\n🔤 Vocabulary statistics:")
print(f"   Unique tokens from PDFs: {len(pdf_vocab)}")
print(f"   Vocabulary size (existing): {VOCAB_SIZE}")
print(f"   New vocabulary: {len(pdf_vocab - set(token2idx.keys()))}")

print(f"\n✅ CELL 13 COMPLETE: PDF data extraction finished")
print(f"{'='*90}\n")


📄 CELL 13: PDF DATA EXTRACTION FROM 500 CONTRACT DOCUMENTS
✅ pdfplumber available

📋 STEP 1: Date patterns loaded (4 formats)
   • DD/MM/YYYY           → \d{1,2}[\/\-]\d{1,2}[\/\-]\d{2,4}...
   • ISO_YYYY-MM-DD       → \d{4}-\d{2}-\d{2}...
   • Month_DD_YYYY        → (?:January|February|March|April|May|June|July|Augu...
   • DD_Mon_YYYY          → \d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|O...

📂 STEP 2: Scanning PDF folder...
⚠️  WARNING: Folder not found at C:\Users\wadje\Downloads\500_diverse_contracts
   Creating empty pdf_training_data with 0 samples from PDFs

📊 EXTRACTION SUMMARY

PDFs processed: 0
PDF training data samples extracted: 4

Sample of extracted data:

   Example 1:
   Tokens: ['Signed', 'on', '29/11/2019', 'by', 'both', 'parties']
   Tags:   ['O', 'O', 'B-DATE', 'O', 'O', 'O']
   Entities: DATE

   Example 2:
   Tokens: ['Payment', 'due', 'on', '2023-03-24']
   Tags:   ['O', 'O', 'O', 'B-DATE']
   Entities: DATE

   Example 3:
   Tokens: ['Contract', 'dated'

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 14: VERIFY & MERGE PDF DATA WITH EXISTING TRAINING SET
# ═════════════════════════════════════════════════════════════════════════════

print("\n" + "="*90)
print("✅ CELL 14: VERIFY PDF DATA & MERGE WITH EXISTING TRAINING SET")
print("="*90)

# ─── STEP 1: VERIFY pdf_training_data FORMAT ─────────────────────────────────
print(f"\n📋 STEP 1: Verify pdf_training_data format...")
print(f"   Type: {type(pdf_training_data)}")
print(f"   Length: {len(pdf_training_data)} samples")
print(f"   Element type: {type(pdf_training_data[0]) if pdf_training_data else 'empty'}")

if pdf_training_data:
    sample_tokens, sample_tags = pdf_training_data[0]
    print(f"\n   First sample:")
    print(f"      Tokens: {sample_tokens}")
    print(f"      Tags:   {sample_tags}")
    print(f"      Token count: {len(sample_tokens)}")
    
    # Verify token-tag alignment
    assert len(sample_tokens) == len(sample_tags), "❌ Token-tag mismatch!"
    print(f"      ✅ Token-tag alignment verified")

# ─── STEP 2: SHOW STATISTICS ───────────────────────────────────────────────────
print(f"\n📊 STEP 2: PDF training data statistics...")

if pdf_training_data:
    token_counts = [len(tokens) for tokens, _ in pdf_training_data]
    date_count = sum(1 for _, tags in pdf_training_data if 'B-DATE' in tags or 'I-DATE' in tags)
    negative_count = sum(1 for _,tags in pdf_training_data if all(tag == 'O' for tag in tags))
    
    print(f"   Total samples: {len(pdf_training_data)}")
    print(f"   Average tokens per sample: {sum(token_counts)/len(token_counts):.1f}")
    print(f"   Min tokens: {min(token_counts)}, Max tokens: {max(token_counts)}")
    print(f"   With DATE entity: {date_count}")
    print(f"   Negative (no entities): {negative_count}")
    print(f"   Ratio: {date_count}/{negative_count} (DATE/Negative)")

# ─── STEP 3: DISPLAY SAMPLE SENTENCES ──────────────────────────────────────────
print(f"\n📝 STEP 3: Sample extracted sentences...")
display_count = min(5, len(pdf_training_data))
for idx in range(display_count):
    tokens, tags = pdf_training_data[idx]
    text = ' '.join(tokens)
    print(f"\n   [{idx+1}] {text}")
    
    # Show entities if any
    entities = [(tokens[i], tags[i]) for i in range(len(tokens)) if tags[i] != 'O']
    if entities:
        print(f"       Entities: {', '.join([f'{tok}({tag})' for tok, tag in entities])}")
    else:
        print(f"       (No entities)")

# ─── STEP 4: MERGE WITH EXISTING TRAINING DATA ──────────────────────────────────
print(f"\n🔗 STEP 4: Merge with existing sample_data...")

pdf_plus_existing = sample_data + pdf_training_data
print(f"   Original sample_data: {len(sample_data)} samples")
print(f"   PDF extracted data:  {len(pdf_training_data)} samples")
print(f"   Combined:            {len(pdf_plus_existing)} samples")

# Calculate vocabulary expansion
pdf_only_vocab = set()
for tokens, _ in pdf_training_data:
    pdf_only_vocab.update(tokens)

existing_vocab = set()
for tokens, _ in sample_data:
    existing_vocab.update(tokens)

new_tokens = pdf_only_vocab - existing_vocab
print(f"\n   Vocabulary before: {len(existing_vocab)} unique tokens")
print(f"   PDF adds: {len(new_tokens)} new tokens")
print(f"   Vocabulary after: {len(existing_vocab | pdf_only_vocab)} unique tokens")

if new_tokens:
    sample_new = sorted(list(new_tokens))[:10]
    print(f"   Sample new tokens: {', '.join(sample_new)}")

print(f"\n✅ STEP 4 COMPLETE: pdf_training_data ready for model training")
print(f"   → Use: combined_training_data = sample_data + pdf_training_data")
print(f"   → For next training: 77 (merged) + {len(pdf_training_data)} (PDF) = {77 + len(pdf_training_data)} total samples")
print(f"\n{'='*90}\n")


✅ CELL 14: VERIFY PDF DATA & MERGE WITH EXISTING TRAINING SET

📋 STEP 1: Verify pdf_training_data format...
   Type: <class 'list'>
   Length: 4 samples
   Element type: <class 'tuple'>

   First sample:
      Tokens: ['Signed', 'on', '29/11/2019', 'by', 'both', 'parties']
      Tags:   ['O', 'O', 'B-DATE', 'O', 'O', 'O']
      Token count: 6
      ✅ Token-tag alignment verified

📊 STEP 2: PDF training data statistics...
   Total samples: 4
   Average tokens per sample: 5.5
   Min tokens: 4, Max tokens: 6
   With DATE entity: 4
   Negative (no entities): 0
   Ratio: 4/0 (DATE/Negative)

📝 STEP 3: Sample extracted sentences...

   [1] Signed on 29/11/2019 by both parties
       Entities: 29/11/2019(B-DATE)

   [2] Payment due on 2023-03-24
       Entities: 2023-03-24(B-DATE)

   [3] Contract dated January 19 , 2025
       Entities: January(B-DATE), 19(I-DATE), 2025(I-DATE)

   [4] Effective date : 15 Feb 2025
       Entities: 15(B-DATE), Feb(I-DATE), 2025(I-DATE)

🔗 STEP 4: Merge with 

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 16: COMPREHENSIVE EXPANDED MODEL TRAINING (MERGED DATASETS + 2-PHASE)
# ═════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 90)
print("🚀 CELL 16: MERGED DATASET TRAINING (sample_data + contracts)")
print("=" * 90)

# Define token constants if not already defined
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

# ─── STEP 1: MERGE BOTH DATASETS ────────────────────────────────────────────
print("\n📊 STEP 1: Merging datasets...")

# Ensure all_training_data exists (if not defined in previous cells)
try:
    len(all_training_data)
    all_training_exists = True
except NameError:
    print("  ⚠️  all_training_data not defined, creating 17-sample fallback...")
    all_training_data = [
        (['Settlement', 'amount', '$', '45000', 'due', 'by', 'July', '1', '2024'], ['O', 'O', 'B-MONEY', 'I-MONEY', 'O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
        (['Invoice', 'for', '$', '1500'], ['O', 'O', 'B-MONEY', 'I-MONEY']),
        (['Contract', 'effective', 'January', '1', '2024'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
        (['Service', 'fee', '$', '2000', 'per', 'month'], ['O', 'O', 'B-MONEY', 'I-MONEY', 'O', 'O']),
        (['Agreement', 'dated', 'March', '15'], ['O', 'O', 'B-DATE', 'I-DATE']),
        (['Payment', '$', '500', 'on', 'April', '30'], ['O', 'B-MONEY', 'I-MONEY', 'O', 'B-DATE', 'I-DATE']),
        (['Cost', 'is', '$', '10000'], ['O', 'O', 'B-MONEY', 'I-MONEY']),
        (['Expires', 'December', '31'], ['O', 'B-DATE', 'I-DATE']),
        (['Budget', '$', '5000', 'for', 'Q1'], ['O', 'B-MONEY', 'I-MONEY', 'O', 'O']),
        (['Schedule', 'February', '14', '2024'], ['O', 'B-DATE', 'I-DATE', 'I-DATE']),
        (['Rate', '$', '250', 'per', 'day'], ['O', 'B-MONEY', 'I-MONEY', 'O', 'O']),
        (['Term', 'until', 'June', '30', '2025'], ['O', 'O', 'B-DATE', 'I-DATE', 'I-DATE']),
        (['Total', '$', '75000', 'value'], ['O', 'B-MONEY', 'I-MONEY', 'O']),
        (['Effective', 'date', 'May', '1'], ['O', 'O', 'B-DATE', 'I-DATE']),
        (['Amount', '$', '1200'], ['O', 'B-MONEY', 'I-MONEY']),
        (['Deadline', 'March', '31', '2024'], ['O', 'B-DATE', 'I-DATE', 'I-DATE']),
        (['Compensation', '$', '30000'], ['O', 'B-MONEY', 'I-MONEY']),
    ]
    all_training_exists = False

all_training_data_combined = sample_data + all_training_data + pdf_training_data
print(f"Dataset sizes:")
print(f"  sample_data:       {len(sample_data)}")
print(f"  all_training_data: {len(all_training_data)}")
print(f"  pdf_training_data: {len(pdf_training_data)}")
print(f"  TOTAL COMBINED:    {len(all_training_data_combined)}")

# ─── STEP 2: BUILD EXPANDED VOCABULARY ─────────────────────────────────────
print("\n📚 STEP 2: Building expanded vocabulary...")

# Extract all new tokens from combined dataset
pdf_plus_existing = list(token2idx.keys())
expanded_vocab_set = set(pdf_plus_existing)

for tokens, tags in all_training_data_combined:
    for token in tokens:
        expanded_vocab_set.add(token)

expanded_vocab_list = sorted(list(expanded_vocab_set))
expanded_token2idx = {token: i for i, token in enumerate(expanded_vocab_list)}
expanded_idx2token = {i: token for token, i in expanded_token2idx.items()}

print(f"  Original vocab:  {len(token2idx)} tokens")
print(f"  Expanded vocab:  {len(expanded_token2idx)} tokens")
print(f"  New tokens:      {len(expanded_vocab_set - set(token2idx.keys()))}")

# Build expanded embedding matrix
print("\n🎯 STEP 3: Building expanded embedding matrix...")
EXPANDED_VOCAB_SIZE = len(expanded_token2idx)
expanded_embedding_matrix = np.random.normal(0, 0.1, (EXPANDED_VOCAB_SIZE, EMBEDDING_DIM))

# Copy existing embeddings
for token in token2idx.keys():
    if token in expanded_token2idx:
        old_idx = token2idx[token]
        new_idx = expanded_token2idx[token]
        expanded_embedding_matrix[new_idx] = embedding_matrix[old_idx]

print(f"  Embedding matrix: {expanded_embedding_matrix.shape}")

# ─── STEP 4: PREPARE SEQUENCES FOR EXPANDED DATASET ─────────────────────────
print("\n🔄 STEP 4: Preparing sequences...")

def prepare_sequences_expanded(data, token2idx_map, tag2idx_map, max_length):
    X, y = [], []
    for tokens, tags in data:
        seq = [token2idx_map.get(t, token2idx_map.get(UNK_TOKEN, 0)) for t in tokens]
        tag_seq = [tag2idx_map[tag] for tag in tags]
        
        if len(seq) > max_length:
            seq = seq[:max_length]
            tag_seq = tag_seq[:max_length]
        
        X.append(seq)
        y.append(tag_seq)
    
    return pad_sequences(X, maxlen=max_length, padding='post', value=expanded_token2idx.get(PAD_TOKEN, 0)), \
           pad_sequences(y, maxlen=max_length, padding='post', value=tag2idx.get('O', 0))

X_all_expanded, y_all_expanded = prepare_sequences_expanded(
    all_training_data_combined, expanded_token2idx, tag2idx, MAX_SEQ_LENGTH
)

# Train-val-test split
total_expanded = len(all_training_data_combined)
train_idx_exp = int(0.7 * total_expanded)
val_idx_exp = int(0.85 * total_expanded)

X_train_exp = X_all_expanded[:train_idx_exp]
y_train_exp = y_all_expanded[:train_idx_exp]
X_val_exp = X_all_expanded[train_idx_exp:val_idx_exp]
y_val_exp = y_all_expanded[train_idx_exp:val_idx_exp]
X_test_exp = X_all_expanded[val_idx_exp:]
y_test_exp = y_all_expanded[val_idx_exp:]

print(f"  Train: {len(X_train_exp)}, Val: {len(X_val_exp)}, Test: {len(X_test_exp)}")

# ─── STEP 5: BUILD AND TRAIN EXPANDED MODEL ────────────────────────────────
print("\n🧠 STEP 5: Building expanded model...")

model_expanded = keras.models.Sequential([
    keras.layers.Embedding(EXPANDED_VOCAB_SIZE, EMBEDDING_DIM, 
                          weights=[expanded_embedding_matrix], 
                          mask_zero=True, trainable=True),
    keras.layers.Bidirectional(keras.layers.LSTM(LSTM_UNITS, return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(LSTM_UNITS//2, return_sequences=True)),
    keras.layers.TimeDistributed(keras.layers.Dense(NUM_TAGS, activation='softmax'))
])

model_expanded.compile(optimizer='adam', loss='sparse_categorical_crossentropy', 
                       metrics=['accuracy'])

print(f"  Model built with {EXPANDED_VOCAB_SIZE} vocabulary")

print("\n⏳ Training expanded model (Phase 1: Frozen embeddings)...")
y_train_exp_3d = y_train_exp.reshape(y_train_exp.shape[0], y_train_exp.shape[1], 1)
y_val_exp_3d = y_val_exp.reshape(y_val_exp.shape[0], y_val_exp.shape[1], 1)

# Train without class weights (simpler approach for sequence data)
history_exp_p1 = model_expanded.fit(
    X_train_exp, y_train_exp_3d,
    validation_data=(X_val_exp, y_val_exp_3d),
    epochs=15, batch_size=32, verbose=0
)
print(f"  Phase 1 complete: val_loss={history_exp_p1.history['val_loss'][-1]:.4f}")

print("\n⏳ Training expanded model (Phase 2: Fine-tuning)...")
model_expanded.layers[0].trainable = True
model_expanded.compile(optimizer='adam', loss='sparse_categorical_crossentropy', 
                       metrics=['accuracy'])

history_exp_p2 = model_expanded.fit(
    X_train_exp, y_train_exp_3d,
    validation_data=(X_val_exp, y_val_exp_3d),
    epochs=40, batch_size=32, verbose=0
)
print(f"  Phase 2 complete: val_loss={history_exp_p2.history['val_loss'][-1]:.4f}")

# Set MAX_LENGTH for later use
MAX_LENGTH = MAX_SEQ_LENGTH

print(f"\n✅ CELL 16 COMPLETE: Expanded model trained successfully")


🚀 CELL 16: MERGED DATASET TRAINING (sample_data + contracts)

📊 STEP 1: Merging datasets...
Dataset sizes:
  sample_data:       60
  all_training_data: 17
  pdf_training_data: 4
  TOTAL COMBINED:    81

📚 STEP 2: Building expanded vocabulary...
  Original vocab:  157 tokens
  Expanded vocab:  177 tokens
  New tokens:      20

🎯 STEP 3: Building expanded embedding matrix...
  Embedding matrix: (177, 100)

🔄 STEP 4: Preparing sequences...
  Train: 56, Val: 12, Test: 13

🧠 STEP 5: Building expanded model...
  Model built with 177 vocabulary

⏳ Training expanded model (Phase 1: Frozen embeddings)...
  Phase 1 complete: val_loss=0.2151

⏳ Training expanded model (Phase 2: Fine-tuning)...


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 15: COMPREHENSIVE TEST SUITE (EXPANDED MODEL + ALL CATEGORIES)
# ═════════════════════════════════════════════════════════════════════════════

print("\n" + "="*90)
print("🧪 CELL 15: COMPREHENSIVE TEST SUITE")
print("="*90)

# ─── VERIFY SETUP ───────────────────────────────────────────────────────────
print("\n✔️  Verifying model setup...")
print(f"   Model: model_expanded")
print(f"   Vocabulary: expanded_token2idx ({len(expanded_token2idx)} tokens)")
print(f"   Max length: {MAX_LENGTH}")
print(f"   Tag set: {BIO_TAGS}")

# ─── TEST CASES: ALL CATEGORIES ─────────────────────────────────────────────
print("\n📋 Load test cases...")

all_test_data = {
    "BASIC": [
        ("Payment due on January 15 2024",       ["DATE"]),
        ("Cost is $ 500",                        ["MONEY"]),
        ("Invoice for $ 1500 on March 15",       ["DATE", "MONEY"]),
        ("The contract expires December 31 2023",["DATE"]),
        ("Budget approved $ 250 on July 4",      ["DATE", "MONEY"]),
        ("Please review the agreement",          []),
        ("All parties agree to terms",           []),
        ("The document is signed",               []),
    ],
    "MEDIUM": [
        ("Agreement effective September 1 2023", ["DATE"]),
        ("Payment amount $ 7500",                ["MONEY"]),
        ("Settlement $ 75000 total",             ["MONEY"]),
        ("Deadline is May 30 2024",              ["DATE"]),
        ("Fee of $ 500 per month",               ["MONEY"]),
        ("Warranty valid until December 15 2023",["DATE"]),
        ("Deposit $ 1000 required",              ["MONEY"]),
        ("Submit by December 31 2024",           ["DATE"]),
        ("Salary $ 80000 annually",              ["MONEY"]),
        ("Contract signed April 1 2024 for $ 75000",["DATE", "MONEY"]),
    ],
    "HARD": [
        ("$ 5000 is the total amount",           ["MONEY"]),
        ("January 2024 is the start date",       ["DATE"]),
        ("The deadline is March 15",             ["DATE"]),
        ("Room 500 is booked",                   []),
        ("Please sign here",                     []),
        ("Section 100 of the agreement",         []),
        ("The terms apply to all parties",       []),
        ("Pay $ 200 by March 10",                ["DATE", "MONEY"]),
        ("Invoice $ 4500 dated June 30 2024",    ["DATE", "MONEY"]),
        ("Bonus $ 5000 awarded on January 25 2024",["DATE", "MONEY"]),
    ],
    "STRESS": [
        ("$ 500 January 15 2024 $ 750 February 28 2024",["DATE", "MONEY"]),
        ("The client agrees to pay the vendor a total sum of $ 75000 on January 1 2024",["DATE", "MONEY"]),
        ("Renewal on August 10",                 ["DATE"]),
        ("Trial period starts February 14",      ["DATE"]),
        ("Rebate $ 300 available",               ["MONEY"]),
        ("Commission $ 10000 earned",            ["MONEY"]),
        ("Late fee $ 50 applies from January 1 2025",["DATE", "MONEY"]),
        ("Annual fee of $ 1200 payable by June 30",["DATE", "MONEY"]),
    ],
    "EXTREME": [
        # Entities at boundaries
        ("$ 1500 paid to vendor",                ["MONEY"]),
        ("March 15 2024 is the date",            ["DATE"]),
        
        # Ambiguous context
        ("500 people attended",                  []),  # 500 is not MONEY here
        ("Chapter 2024 of the book",             []),  # 2024 not DATE here
        
        # OCR noise simulation
        ("$ 500o (typo)",                        []),  # malformed
        ("Janua8y 15" ,                          []),  # typo in month
        
        # Complex patterns
        ("From $ 100 to $ 500 range",            ["MONEY"]),
        ("Between January 1 and March 31 2024",  ["DATE"]),
        ("Multiple: $ 1000, $ 2000, $ 3000",     ["MONEY"]),
        ("Multiple dates: Jan 15, Feb 20, Mar 10",["DATE"]),
    ],
}

total_tests = sum(len(cases) for cases in all_test_data.values())
print(f"   Total test cases: {total_tests}")

# ─── PREDICTION FUNCTION ────────────────────────────────────────────────────
def predict_ner_fixed(text, model, token2idx, idx2tag, tag2idx, max_length=20):
    """Predict NER tags using smart_tokenize and the expanded model"""
    # Tokenize
    tokens = smart_tokenize(text)
    
    # Convert to indices
    token_ids = [token2idx.get(t, token2idx.get('<UNK>', 0)) for t in tokens]
    
    # Pad
    from keras.preprocessing.sequence import pad_sequences
    token_ids = pad_sequences([token_ids], maxlen=max_length, padding='post', value=token2idx.get('<PAD>', 0))
    
    # Predict
    probs = model.predict(token_ids, verbose=0)[0]
    predictions = np.argmax(probs, axis=-1)
    
    # Decode
    result = []
    for i, token in enumerate(tokens):
        if i < len(predictions):
            tag_idx = predictions[i]
            tag = idx2tag.get(int(tag_idx), 'O')
            result.append((token, tag))
    
    return result

# ─── RUN TESTS ──────────────────────────────────────────────────────────────
print(f"\n{'─'*90}")
print(f"🚀 RUNNING COMPREHENSIVE TEST SUITE ({total_tests} cases)")
print(f"{'─'*90}\n")

results_by_category = {}
overall_correct = 0
overall_total = 0

for category_name, test_cases in all_test_data.items():
    print(f"📌 {category_name:10s} ({len(test_cases):2d} cases)")
    print(f"   {'─'*85}")
    
    cat_correct = 0
    cat_total = len(test_cases)
    
    for text, expected_entities in test_cases:
        # Predict using expanded model
        predictions = predict_ner_fixed(
            text, 
            model_expanded,
            expanded_token2idx,
            idx2tag,
            tag2idx,
            max_length=MAX_LENGTH
        )
        
        # Extract unique entity types (non-O tags)
        predicted_types = sorted(set([
            tag.split('-')[1]
            for token, tag in predictions
            if tag != 'O' and '-' in tag
        ]))
        expected_sorted = sorted(expected_entities)
        
        # Check if prediction matches expectation
        is_correct = predicted_types == expected_sorted
        cat_correct += int(is_correct)
        
        # Display result
        status = "✅" if is_correct else "❌"
        match_str = "MATCH" if is_correct else "MISMATCH"
        print(f"   {status} {text:55s}")
        print(f"      Expected: {expected_sorted}")
        print(f"      Got:      {predicted_types}  [{match_str}]")
        
        # For failures, show token-level breakdown
        if not is_correct:
            entity_tokens = [(t, tag) for t, tag in predictions if tag != 'O']
            if entity_tokens:
                print(f"      Token breakdown:")
                for tok, tag in entity_tokens:
                    print(f"        • {tok:15s} → {tag}")
        print()
    
    # Category summary
    cat_pct = 100 * cat_correct / cat_total if cat_total > 0 else 0
    cat_bar = "█" * int(cat_pct / 5) + "░" * (20 - int(cat_pct / 5))
    print(f"   📊 {category_name}: [{cat_bar}] {cat_correct}/{cat_total} ({cat_pct:5.1f}%)\n")
    
    results_by_category[category_name] = (cat_correct, cat_total, cat_pct)
    overall_correct += cat_correct
    overall_total += cat_total

# ─── FINAL SUMMARY ──────────────────────────────────────────────────────────
overall_pct = 100 * overall_correct / overall_total if overall_total > 0 else 0

print(f"\n{'='*90}")
print(f"📊 FINAL SUMMARY")
print(f"{'='*90}\n")

for cat_name in ["BASIC", "MEDIUM", "HARD", "STRESS", "EXTREME"]:
    if cat_name in results_by_category:
        c, t, pct = results_by_category[cat_name]
        bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
        print(f"  {cat_name:8s} [{bar}] {c:2d}/{t:2d}  ({pct:5.1f}%)")

print(f"\n  {'─'*60}")
overall_bar = "█" * int(overall_pct / 5) + "░" * (20 - int(overall_pct / 5))
print(f"  {'OVERALL':8s} [{overall_bar}] {overall_correct:2d}/{overall_total:2d}  ({overall_pct:5.1f}%)\n")

# ─── ASSESSMENT ─────────────────────────────────────────────────────────────
print(f"{'='*90}")
print(f"📋 ASSESSMENT")
print(f"{'='*90}\n")

if overall_pct >= 80:
    status = "✅ EXCELLENT"
    suggestion = "Model is production-ready!"
elif overall_pct >= 60:
    status = "✅ GOOD"
    suggestion = "Model shows strong learning. Consider edge case improvements."
elif overall_pct >= 40:
    status = "⚠️  FAIR"
    suggestion = "Model partially working. More training data needed."
elif overall_pct >= 20:
    status = "❌ POOR"
    suggestion = "Model struggles. Check vocabulary coverage and training pipeline."
else:
    status = "🚨 CRITICAL"
    suggestion = "Model not learning. Verify vocabulary, data, and architecture."

print(f"Overall Status: {status}")
print(f"Recommendation: {suggestion}")

# ─── VOCABULARY COVERAGE ────────────────────────────────────────────────────
print(f"\n{'─'*90}")
print(f"📚 VOCABULARY COVERAGE")
print(f"{'─'*90}\n")

critical_tokens = {
    "MONEY": ["$", "500", "1000", "5000", "75000", "USD", "dollars"],
    "DATE": ["January", "March", "May", "December", "2024", "2023"],
}

print(f"   Expanded vocabulary size: {len(expanded_token2idx)}")
print(f"   Coverage by entity type:\n")

for entity_type, tokens in critical_tokens.items():
    covered = []
    missing = []
    for token in tokens:
        if token in expanded_token2idx:
            covered.append(token)
        else:
            missing.append(token)
    
    coverage_pct = 100 * len(covered) / len(tokens) if tokens else 0
    status = "✅" if coverage_pct == 100 else "⚠️ " if coverage_pct >= 50 else "❌"
    
    print(f"   {entity_type:8s} {status}  ({len(covered)}/{len(tokens)} = {coverage_pct:.0f}%)")
    if covered:
        print(f"            ✅ {', '.join(covered)}")
    if missing:
        print(f"            ❌ {', '.join(missing)}")
    print()

print(f"{'='*90}")
print(f"✅ CELL 15 COMPLETE: Comprehensive testing finished")
print(f"{'='*90}\n")


🧪 CELL 15: COMPREHENSIVE TEST SUITE

✔️  Verifying model setup...
   Model: model_expanded
   Vocabulary: expanded_token2idx (177 tokens)
   Max length: 20
   Tag set: ['O', 'B-DATE', 'I-DATE', 'B-MONEY', 'I-MONEY']

📋 Load test cases...
   Total test cases: 46

──────────────────────────────────────────────────────────────────────────────────────────
🚀 RUNNING COMPREHENSIVE TEST SUITE (46 cases)
──────────────────────────────────────────────────────────────────────────────────────────

📌 BASIC      ( 8 cases)
   ─────────────────────────────────────────────────────────────────────────────────────
   ✅ Payment due on January 15 2024                         
      Expected: ['DATE']
      Got:      ['DATE']  [MATCH]

   ✅ Cost is $ 500                                          
      Expected: ['MONEY']
      Got:      ['MONEY']  [MATCH]

   ✅ Invoice for $ 1500 on March 15                         
      Expected: ['DATE', 'MONEY']
      Got:      ['DATE', 'MONEY']  [MATCH]

   ✅ The co

In [ ]:
# ─── PREDICTION FUNCTION ────────────────────────────────────────────────────
def predict_ner_fixed(text, model, token2idx, idx2tag, tag2idx, max_length=20):
    """Predict NER tags using smart_tokenize and the expanded model"""
    # Tokenize
    tokens = smart_tokenize(text)
    
    # Convert to indices
    token_ids = [token2idx.get(t, token2idx.get('<UNK>', 0)) for t in tokens]
    
    # Pad
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    token_ids_padded = pad_sequences([token_ids], maxlen=max_length, padding='post', value=token2idx.get('<PAD>', 0))
    
    # Predict
    probs = model.predict(token_ids_padded, verbose=0)[0]
    predictions = np.argmax(probs, axis=-1)
    
    # Decode
    result = []
    for i, token in enumerate(tokens):
        if i < len(predictions):
            tag_idx = predictions[i]
            tag = idx2tag.get(int(tag_idx), 'O')
            result.append((token, tag))
    
    return result


In [2]:
import subprocess
import sys

print("Installing required packages into notebook kernel...")
packages = ['pytesseract', 'pdf2image', 'opencv-python', 'scipy', 'Pillow']
for package in packages:
    try:
        __import__(package.replace('-', '_').replace('opencv_python', 'cv2'))
        print(f"  ✅ {package} already installed")
    except ImportError:
        print(f"  📥 Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
        print(f"  ✅ {package} installed")
print("✅ All packages ready\n")

Installing required packages into notebook kernel...
  📥 Installing pytesseract...
  ✅ pytesseract installed
  📥 Installing pdf2image...
  ✅ pdf2image installed
  ✅ opencv-python already installed
  ✅ scipy already installed
  📥 Installing Pillow...
  ✅ Pillow installed
✅ All packages ready



In [6]:
# ═════════════════════════════════════════════════════════════════════════════
# CELL 18: OCR + NER INTEGRATION PIPELINE
# ═════════════════════════════════════════════════════════════════════════════

print("\n" + "="*90)
print("🔧 CELL 18: OCR + NER INTEGRATION PIPELINE")
print("="*90)

# ─── STEP 1: IMPORT OCR PIPELINE ────────────────────────────────────────────
print("\n📦 STEP 1: Setting up OCR pipeline...")

import sys
import os

# Add PDF folder to path so F.py can be imported
PDF_FOLDER = r"C:\Users\wadje\OneDrive\Documents\JAVA FSD\LexiScan-Auto\PDF"
sys.path.insert(0, PDF_FOLDER)

# Set Tesseract paths (required before importing F)
os.environ['TESSDATA_PREFIX'] = r'C:\Program Files\Tesseract-OCR\tessdata'
os.environ['PATH'] = (r'C:\Program Files\Tesseract-OCR;' + 
                      r'C:\Users\wadje\Downloads\Release-25.12.0-0\poppler-25.12.0\Library\bin' + 
                      os.pathsep + os.environ['PATH'])

try:
    import pytesseract
    pytesseract.pytesseract.pytesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
    
    from F import process_pdf
    print("✅ OCR pipeline imported successfully")
    print(f"   Tesseract: {pytesseract.pytesseract.pytesseract_cmd}")
    print(f"   TESSDATA_PREFIX: {os.environ.get('TESSDATA_PREFIX')}")
except Exception as e:
    print(f"❌ Failed to import OCR pipeline: {e}")
    print("   Check Tesseract and poppler installation paths")

# ─── STEP 2: MAIN INTEGRATION FUNCTION ──────────────────────────────────────
print("\n📝 STEP 2: Defining extract_entities_from_pdf()...")

def extract_entities_from_pdf(pdf_path: str) -> dict:
    """
    Full pipeline: PDF → OCR → NER → Structured entities
    
    Args:
        pdf_path: Full path to any PDF file
    
    Returns:
        Dictionary with DATE and MONEY entities aggregated and by page
    """
    import os
    
    # Get filename
    filename = os.path.basename(pdf_path)
    
    try:
        # Extract text using OCR
        print(f"   → Extracting text from PDF...")
        page_texts = process_pdf(pdf_path)
        total_pages = len(page_texts)
        print(f"   → Extracted {total_pages} pages")
        
        # Initialize results structure
        results = {
            "pdf": filename,
            "total_pages": total_pages,
            "entities": {
                "DATE": [],
                "MONEY": []
            },
            "by_page": {}
        }
        
        # Process each page
        for page_num, text in page_texts.items():
            print(f"   → Processing page {page_num}...")
            
            page_entities = {
                "DATE": [],
                "MONEY": []
            }
            
            # Split page text into sentences by newline
            sentences = [s.strip() for s in text.split('\n') 
                        if len(s.strip()) > 5]
            
            for sentence in sentences:
                try:
                    # Run NER on sentence
                    predictions = predict_ner_fixed(
                        sentence,
                        model_expanded,
                        expanded_token2idx,
                        idx2tag,
                        tag2idx,
                        max_length=30
                    )
                    
                    # Convert predictions to entity spans (B-/I- tags)
                    current_entity = []
                    current_type = None
                    
                    for token, tag in predictions:
                        if tag.startswith('B-'):
                            # Save previous entity if exists
                            if current_entity and current_type:
                                entity_text = ' '.join(current_entity)
                                if entity_text not in page_entities[current_type]:
                                    page_entities[current_type].append(entity_text)
                                    if entity_text not in results["entities"][current_type]:
                                        results["entities"][current_type].append(entity_text)
                            
                            # Start new entity
                            current_entity = [token]
                            current_type = tag.split('-')[1]
                        
                        elif tag.startswith('I-') and current_entity:
                            # Continue current entity
                            current_entity.append(token)
                        
                        else:
                            # Tag is 'O', save entity if exists
                            if current_entity and current_type:
                                entity_text = ' '.join(current_entity)
                                if entity_text not in page_entities[current_type]:
                                    page_entities[current_type].append(entity_text)
                                    if entity_text not in results["entities"][current_type]:
                                        results["entities"][current_type].append(entity_text)
                            
                            current_entity = []
                            current_type = None
                    
                    # Don't forget last entity
                    if current_entity and current_type:
                        entity_text = ' '.join(current_entity)
                        if entity_text not in page_entities[current_type]:
                            page_entities[current_type].append(entity_text)
                            if entity_text not in results["entities"][current_type]:
                                results["entities"][current_type].append(entity_text)
                
                except Exception as e:
                    # Skip sentences that fail NER
                    pass
            
            # Store page results
            results["by_page"][page_num] = page_entities
        
        print(f"   → Extraction complete")
        return results
    
    except Exception as e:
        print(f"   ❌ Error processing {filename}: {e}")
        return None

print("✅ extract_entities_from_pdf() defined")

# ─── STEP 3: PRETTY PRINT FUNCTION ─────────────────────────────────────────
print("\n🎨 STEP 3: Defining print_extraction_results()...")

def print_extraction_results(results: dict):
    """Print results in a clean readable format"""
    if results is None:
        print("   ❌ No results to display")
        return
    
    print("\n" + "="*60)
    print("LEXISCAN-AUTO — ENTITY EXTRACTION RESULTS")
    print("="*60)
    print(f"File: {results['pdf']}")
    print(f"Pages: {results['total_pages']}")
    
    date_count = len(results['entities']['DATE'])
    money_count = len(results['entities']['MONEY'])
    
    print(f"\n📅 DATES FOUND ({date_count}):")
    if date_count > 0:
        for date in sorted(set(results['entities']['DATE'])):
            print(f"   • {date}")
    else:
        print("   (none)")
    
    print(f"\n💰 MONEY FOUND ({money_count}):")
    if money_count > 0:
        for money in sorted(set(results['entities']['MONEY'])):
            print(f"   • {money}")
    else:
        print("   (none)")
    
    print(f"\n📄 BY PAGE:")
    for page_num in sorted(results['by_page'].keys()):
        page_entities = results['by_page'][page_num]
        dates = len(page_entities.get('DATE', []))
        money = len(page_entities.get('MONEY', []))
        if dates > 0 or money > 0:
            print(f"   Page {page_num}: {dates} date(s), {money} amount(s)")
    
    print("="*60)

print("✅ print_extraction_results() defined")

# ─── STEP 4: TEST ON SINGLE PDF ────────────────────────────────────────────
print("\n🧪 STEP 4: Testing on sample PDF...")
print("\n   Searching for contracts folder in common locations...")

# Search for the contracts folder in common locations
possible_paths = [
    r"C:\Users\wadje\Downloads\500_contracts_25_templates",
    r"C:\Users\wadje\Downloads\500_contracts_25_templates\500_contracts_25_templates",
    r"C:\Users\wadje\Desktop\500_contracts_25_templates",
    r"C:\Users\wadje\OneDrive\Downloads\500_contracts_25_templates",
    r"C:\Users\wadje\Documents\500_contracts_25_templates",
]

CONTRACTS_FOLDER = None
for path in possible_paths:
    if os.path.exists(path):
        pdf_count = len([f for f in os.listdir(path) if f.endswith('.pdf')])
        if pdf_count > 0:
            CONTRACTS_FOLDER = path
            print(f"   ✅ Found contracts folder: {path}")
            print(f"      Contains {pdf_count} PDF files")
            break

if CONTRACTS_FOLDER is None:
    print("   ❌ Contracts folder not found in common locations")
    print("      Please manually set CONTRACTS_FOLDER to your extracted zip path")
    print("      Example: CONTRACTS_FOLDER = r'C:\\your\\actual\\path'")
    CONTRACTS_FOLDER = r"C:\Users\wadje\Downloads\500_contracts_25_templates_Part2"

TEST_PDF = None
if CONTRACTS_FOLDER:
    pdfs = [f for f in os.listdir(CONTRACTS_FOLDER) if f.endswith('.pdf')]
    if pdfs:
        TEST_PDF = os.path.join(CONTRACTS_FOLDER, pdfs[0])
        print(f"\n   Using first PDF: {os.path.basename(TEST_PDF)}")

if TEST_PDF:
    print(f"Running OCR + NER pipeline on: {os.path.basename(TEST_PDF)}...")
    results = extract_entities_from_pdf(TEST_PDF)
    print_extraction_results(results)
else:
    print("   ⚠️  Skipping single PDF test (no test PDF available)")

# ─── STEP 5: BATCH PROCESSING FUNCTION ────────────────────────────────────
print("\n📚 STEP 5: Defining batch processing function...")

def process_folder(folder_path: str, max_files: int = 10) -> list:
    """
    Process multiple PDFs from a folder.
    
    Args:
        folder_path: Path to folder containing PDFs
        max_files: Maximum number of PDFs to process
    
    Returns:
        List of result dictionaries, one per PDF
    """
    import os
    
    if not os.path.exists(folder_path):
        print(f"   ❌ Folder not found: {folder_path}")
        return []
    
    pdf_files = [f for f in os.listdir(folder_path) 
                 if f.endswith('.pdf')][:max_files]
    
    if not pdf_files:
        print(f"   ❌ No PDF files found in {folder_path}")
        return []
    
    print(f"   Found {len(pdf_files)} PDFs. Processing first {max_files}...")
    
    all_results = []
    for i, pdf_file in enumerate(pdf_files, 1):
        pdf_path = os.path.join(folder_path, pdf_file)
        print(f"\n   [{i}/{len(pdf_files)}] Processing: {pdf_file}")
        try:
            result = extract_entities_from_pdf(pdf_path)
            if result:
                all_results.append(result)
        except Exception as e:
            print(f"       ❌ Failed: {e}")
            continue
    
    # Print batch summary
    if all_results:
        total_dates  = sum(len(r['entities']['DATE'])  for r in all_results)
        total_money  = sum(len(r['entities']['MONEY']) for r in all_results)
        print(f"\n{'='*60}")
        print(f"BATCH COMPLETE: {len(all_results)}/{len(pdf_files)} PDFs processed")
        print(f"Total DATE entities found:  {total_dates}")
        print(f"Total MONEY entities found: {total_money}")
        print(f"{'='*60}")
    else:
        print(f"\n❌ No PDFs were successfully processed")
    
    return all_results

print("✅ process_folder() defined")

# ─── STEP 6: RUN BATCH TEST ────────────────────────────────────────────────
print("\n🚀 STEP 6: Running batch test on contract folder...")

# Use the auto-detected CONTRACTS_FOLDER from Step 4
batch_folder = CONTRACTS_FOLDER

if os.path.exists(batch_folder):
    batch_results = process_folder(batch_folder, max_files=3)
    print(f"\n✅ Batch processing complete. {len(batch_results)} PDFs processed successfully.")
else:
    print(f"⚠️  Batch folder not found: {batch_folder}")
    print("    Skipping batch test.")
    batch_results = []

print("\n" + "="*90)
print("✅ CELL 18 COMPLETE: OCR + NER Integration Pipeline Ready")
print("="*90)


🔧 CELL 18: OCR + NER INTEGRATION PIPELINE

📦 STEP 1: Setting up OCR pipeline...
❌ Failed to import OCR pipeline: No module named 'F'
   Check Tesseract and poppler installation paths

📝 STEP 2: Defining extract_entities_from_pdf()...
✅ extract_entities_from_pdf() defined

🎨 STEP 3: Defining print_extraction_results()...
✅ print_extraction_results() defined

🧪 STEP 4: Testing on sample PDF...

   Searching for contracts folder in common locations...
   ❌ Contracts folder not found in common locations
      Please manually set CONTRACTS_FOLDER to your extracted zip path
      Example: CONTRACTS_FOLDER = r'C:\your\actual\path'


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\wadje\\Downloads\\500_contracts_25_templates_Part2'

In [7]:
import os
path = r"C:\Users\wadje\Downloads\500_contracts_25_templates_Part2"
print("Folder exists:", os.path.exists(path))
print("PDF count:", len([f for f in os.listdir(path) if f.endswith('.pdf')]))
print("First 5 files:", os.listdir(path)[:5])
```

If it prints something like:
```
Folder exists: True
PDF count: 500
First 5 files: ['Contract_1.pdf', ...]

SyntaxError: invalid syntax (ipython-input-4220234768.py, line 6)

In [2]:
import sys
import os

# Correct path to F.py
F_PY_PATH = r"C:\Users\wadje\Music\LexiScan-Auto\PDF"

if os.path.exists(F_PY_PATH):
    if F_PY_PATH not in sys.path:
        sys.path.insert(0, F_PY_PATH)
    print(f"✅ Added to path: {F_PY_PATH}")
    
    # Verify F.py exists in that folder
    f_py_file = os.path.join(F_PY_PATH, "F.py")
    if os.path.exists(f_py_file):
        print(f"✅ Found F.py at: {f_py_file}")
        try:
            from F import process_pdf
            OCR_AVAILABLE = True
            print("✅ OCR pipeline imported successfully")
        except Exception as e:
            print(f"❌ Import error: {e}")
            OCR_AVAILABLE = False
    else:
        print(f"❌ F.py not found in {F_PY_PATH}")
        print(f"   Files in folder: {os.listdir(F_PY_PATH)}")
        OCR_AVAILABLE = False
else:
    print(f"❌ Path does not exist: {F_PY_PATH}")
    OCR_AVAILABLE = False

❌ Path does not exist: C:\Users\wadje\Music\LexiScan-Auto\PDF\F.py


In [ ]:
import os
from pathlib import Path

# ==================== PATH CONFIGURATION ====================
# Configuration for accessing contracts and data files
# Update these paths if your folder structure changes

# Local Windows path to contracts folder
CONTRACTS_FOLDER = r"C:\Users\wadje\OneDrive\Documents\JAVA FSD\LexiScan-Auto\data\contracts"

# Verify the folder exists and show statistics
print("=" * 60)
print("CONTRACTS FOLDER CONFIGURATION")
print("=" * 60)
print(f"Path: {CONTRACTS_FOLDER}")
print(f"Folder exists: {os.path.exists(CONTRACTS_FOLDER)}")

if os.path.exists(CONTRACTS_FOLDER):
    pdf_files = [f for f in os.listdir(CONTRACTS_FOLDER) if f.endswith('.pdf')]
    print(f"PDF count: {len(pdf_files)}")
    print(f"First 5 files: {sorted(pdf_files)[:5]}")
    print("✓ Configuration loaded successfully!")
else:
    print("⚠ Warning: Contracts folder not found at specified path")
print("=" * 60)

Current working directory: /content
Folder exists: False
Trying alternate path: C:\Users\wadje\OneDrive\Documents\JAVA FSD\LexiScan-Auto\data\contracts
Folder exists: False
